# 초간단 RAG 1탄 — 판결서 익명처리 데이터로 판결 이유 초안 받기

익명처리 라벨을 적용한 판결문으로 **내 컴퓨터 안에서만** 도는
RAG 를 만듭니다. 쟁점 한 줄을 넣으면 참고 판결 3건을 찾고,
그 판결을 근거로 가상의 판결 이유 초안을 씁니다.

> **로컬 PC 또는 국내 서버·국내 클라우드에서만 실행하세요.
> Colab 에서는 실행할 수 없습니다.**
> 이 데이터는 AI Hub 약관상 제3자 제공·재배포가 금지되고,
> 해외 서버·클라우드에 전송·저장하거나 그 환경에서 학습하는 것도
> 국외 반출에 해당합니다. 판결문을 외부 API·LLM 으로 보내지 않도록
> 임베딩과 생성 모두 이 컴퓨터의 로컬 모델로 돌립니다.

이 노트북은 과학기술정보통신부와 한국지능정보사회진흥원의 사업결과로
구축된 AI Hub 판결서 익명처리 데이터를 활용했습니다.

| 단계 | 하는 일 | 모델·도구 |
|---|---|---|
| 1 | 데이터 경로 지정 · zip 해제 | 표준 라이브러리 |
| 2 | 라벨 적용 → 가명 코퍼스 | 표준 라이브러리 |
| 3 | 임베딩(캐시 파일 저장) | `BAAI/bge-m3` (MIT) |
| 4 | 쟁점 검색 상위 3건 | 코사인 유사도 |
| 5 | 이유 초안 생성 — 근거 있음/없음 | `Qwen3-4B-Instruct-2507` (Apache-2.0) |

실행 환경은 Apple Silicon Mac 기준입니다. 다른 기기는 5단계에서
transformers 판 셀이 대신 돕니다. 3단계 임베딩이 가장 오래 걸립니다
(M3 Pro 에서 16,800건에 약 1시간). 한 번 만들면 캐시 파일을 씁니다.

생성 결과는 가상의 초안입니다. 법률 자문이 아닙니다.

In [ ]:
import platform
import sys

APPLE = (sys.platform == "darwin"
         and platform.machine() == "arm64")

pkgs = "sentence-transformers numpy"
pkgs += " mlx-lm" if APPLE else " transformers accelerate"
%pip install -q {pkgs}

In [ ]:
import importlib.metadata as md

import torch

print("python      :", platform.python_version())
for name in ["torch", "sentence-transformers",
             "transformers", "numpy"]:
    print(f"{name:<12}:", md.version(name))
if APPLE:
    print("mlx-lm      :", md.version("mlx-lm"))

if torch.cuda.is_available():
    DEV = "cuda"
elif torch.backends.mps.is_available():
    DEV = "mps"
else:
    DEV = "cpu"
print("device      :", DEV)
print("machine     :", platform.platform())

## 1. 데이터 경로 지정

이 저장소의 다른 노트북은 데이터를 실행 시점에 직접 내려받지만,
이 노트북은 예외입니다. 판결서 익명처리 데이터는 **승인제**라
AI Hub 에 로그인해 신청하고 승인받은 사람만 받을 수 있고,
재배포가 금지돼 노트북이 대신 받아 올 수 없습니다.

1. [데이터 페이지](https://aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=71968)
   에서 신청하고 내려받습니다.
2. 압축을 풀면 나오는 `3.개방데이터/2.데이터(NIA)` 폴더 경로를
   아래 `DATA_DIR` 에 적습니다. 안에 zip 8개가 있어야 합니다.

코퍼스·임베딩 같은 중간 파일은 `WORK_DIR` 에 쌓입니다.
이 파일들도 원본과 같은 약관을 따르니 공유하지 마세요.

In [ ]:
import os
from pathlib import Path

# 받은 데이터 폴더 경로를 적으세요. 환경변수로 줘도 됩니다.
DATA_DIR = Path(os.environ.get(
    "AIHUB_JUDGMENT_DIR", "여기에/3.개방데이터/2.데이터(NIA)"))
WORK_DIR = Path("data/legal-rag-01")

zips = sorted(DATA_DIR.expanduser().rglob("*.zip"))
if not zips:
    raise FileNotFoundError(
        "DATA_DIR 에서 zip 을 찾지 못했습니다. "
        "2.데이터(NIA) 폴더 경로를 확인하세요.")
for z in zips:
    mb = z.stat().st_size / 1024 ** 2
    print(f"{z.parent.parent.name:<11}{mb:5.1f}MB  {z.name}")

## 1-1. zip 해제

zip 8개는 원천데이터(TS·VS)와 라벨링데이터(TL·VL)로 나뉩니다.
라벨링 JSON 이 판결문 본문(`sections`)과 익명처리 라벨
(`annotations`)을 함께 갖고 있어 **라벨링데이터만 풀면 됩니다**.

기본은 학습셋(Training) 16,800건입니다. 검증셋 2,100건까지
쓰려면 `SPLITS` 에 `"Validation"` 을 더하세요.

이 zip 은 파일명이 UTF-8 인데 표시 플래그가 없어서, 그대로 풀면
한글 파일명이 깨집니다. 파이썬으로 이름을 바로잡아 풉니다.

In [ ]:
import zipfile

SPLITS = ["Training"]
RAW_DIR = WORK_DIR / "raw"


def fix_name(info):
    if info.flag_bits & 0x800:
        return info.filename
    return info.filename.encode("cp437").decode("utf-8")


for z in zips:
    split = z.parent.parent.name
    if split not in SPLITS or "라벨링" not in z.parent.name:
        continue
    dst = RAW_DIR / z.stem
    if (dst / ".done").exists():
        print("skip   ", z.name)
        continue
    with zipfile.ZipFile(z) as zf:
        for info in zf.infolist():
            name = fix_name(info)
            if name.endswith("/"):
                continue
            out = dst / name
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(zf.read(info))
    (dst / ".done").touch()
    print("unzip  ", z.name)

label_files = sorted(RAW_DIR.rglob("*.json"))
print(f"라벨 파일 {len(label_files):,}건")

## 2. 라벨 적용 → 가명 코퍼스

라벨 하나는 이렇게 생겼습니다(값은 예시).

```json
{"section_id": ["S5"], "span": [{"start": 0, "end": 3}],
 "original_text": "홍길동", "replacement_text": "인물-1"}
```

`section_id` 섹션 본문의 `span` 위치를 `replacement_text` 로
바꿉니다. 실명은 `인물-1`·`원고-2`, 법인명은 `조직-3`, 상세 주소는
`서울특별시 강남구 A` 처럼 시·군·구까지로 일반화됩니다.

세 가지를 보정합니다.

- **span 이 어긋난 라벨** — 약 13% 는 `span` 위치의 글자가
  `original_text` 와 다릅니다. 이런 라벨은 같은 섹션에서
  `original_text` 가 나오는 자리 가운데 **span 에서 가장 가깝고
  아직 다른 라벨이 쓰지 않은 자리 하나**를 고릅니다. 위치를 모두
  정한 뒤 뒤에서부터 바꿔야 앞쪽 위치가 밀리지 않습니다.
  문서 전체를 한꺼번에 문자열 치환하면 안 됩니다 — 원본에서
  이미 `#이름#` 으로 가려진 서로 다른 사람이 첫 라벨의 가명
  하나로 합쳐집니다(`인물-1 … 인물-2` 가 `인물-1 … 인물-1` 로).
  섹션 어디에도 `original_text` 가 없는 라벨(약 2%)은 바꿀 자리가
  없으니 건너뜁니다.
- **라벨 없는 자리에 또 나온 이름** — 한 문서 안에서 가명이
  하나로만 대응하는 원문(2자 이상, `#이름#` 제외)은 남은 자리도
  같은 가명으로 바꿉니다.
- **사건 정보** — 사건명·법원명에도 라벨 대상이 섞여 있어
  (`대법원` 도 조직명으로 라벨돼 있습니다) 같은 치환을 적용합니다.
- **가려진 섹션 제목** — 주문 섹션 제목이 `#이름#` 으로 가려진
  문서가 대부분이라 `주문` 으로 되돌립니다.

검색에 쓸 본문은 사건명·판시사항·판결요지·이유 순으로 이어
앞 1,800자만 씁니다(bge-m3 토큰으로 약 1,000개).

In [ ]:
import json
import unicodedata
from collections import Counter

MAX_CHARS = 1800
CORPUS = WORK_DIR / "corpus.jsonl"


def width(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF"
               else 1 for c in s)


def pad(s, cols):
    return s + " " * (cols - width(s))


def wrap(text, cols=64, indent=""):
    """한글을 2칸으로 세어 cols 칸 안에서 어절 단위 줄바꿈."""
    out = []
    for para in text.split("\n"):
        line = indent
        for word in para.rstrip().split(" "):
            sep = "" if line == indent else " "
            if width(line + sep + word) <= cols:
                line += sep + word
                continue
            if line != indent:
                out.append(line)
            line = indent
            for ch in word:
                if width(line + ch) > cols:
                    out.append(line)
                    line = indent
                line += ch
        out.append(line)
    return "\n".join(out)


MASK = "#이름#"


def locate(text, o, start, used):
    """span 이 맞으면 그 자리, 아니면 가장 가까운 빈 자리."""
    def free(i):
        return all(i + len(o) <= a or i >= b for a, b in used)

    if text[start:start + len(o)] == o and free(start):
        return start, "span"
    best, i = None, text.find(o)
    while i != -1:
        if free(i) and (best is None
                        or abs(i - start) < abs(best - start)):
            best = i
        i = text.find(o, i + 1)
    return best, "near" if best is not None else "miss"


def pseudonymize(doc, stat):
    secs = {s["section_id"]: s["text"]
            for s in doc["sections"]}
    edits, maps = {}, {}
    for a in doc["annotations"]:
        sid, sp = a["section_id"][0], a["span"][0]
        o, r = a["original_text"], a["replacement_text"]
        maps.setdefault(o, set()).add(r)
        es = edits.setdefault(sid, [])
        pos, how = locate(secs[sid], o, sp["start"],
                          [(s, e) for s, e, _ in es])
        stat[how] += 1
        if pos is not None:
            es.append((pos, pos + len(o), r))
    for sid, es in edits.items():
        t = secs[sid]
        for s, e, r in sorted(es, reverse=True):
            t = t[:s] + r + t[e:]
        secs[sid] = t
    sweep = {o: min(rs) for o, rs in maps.items()
             if len(rs) == 1 and len(o) >= 2
             and o != MASK and o not in min(rs)}
    order = sorted(sweep, key=len, reverse=True)

    def clean(text):
        for o in order:
            text = text.replace(o, sweep[o])
        return text

    sections = [("주문" if s["title"] == MASK
                 else s["title"], clean(secs[s["section_id"]]))
                for s in doc["sections"]]
    return sections, clean


def pick(sections, *names):
    for title, text in sections:
        if title.replace(" ", "") in names:
            return text.strip()
    return ""


stat, kinds = Counter(), Counter()
with CORPUS.open("w") as fp:
    for f in label_files:
        doc = json.loads(f.read_text())
        info = doc["info"]
        secs, clean = pseudonymize(doc, stat)
        holding = pick(secs, "판시사항")
        summary = pick(secs, "판결요지")
        reason = pick(secs, "이유")
        head = clean(f"{info['caseNm']} | {info['courtNm']} "
                     f"{info['judmnAdjuDe']} {info['caseNo']}")
        body = "\n".join(x for x in [
            head, holding, summary, reason] if x)
        rec = {"id": f.stem, "kind": info["caseClass"],
               "head": head, "holding": holding,
               "summary": summary, "reason": reason,
               "text": body[:MAX_CHARS],
               "full": "\n\n".join(t for _, t in secs)}
        fp.write(json.dumps(rec, ensure_ascii=False) + "\n")
        kinds[info["caseClass"]] += 1

n_ann = sum(stat.values())
rows = [
    ("라벨 파일", f"{len(label_files):>9,}건"),
    ("라벨", f"{n_ann:>9,}개"),
    ("  span 위치 그대로", f"{stat['span']:>9,}개"),
    ("  가까운 자리로 보정", f"{stat['near']:>9,}개"),
    ("  본문에 원문 없음", f"{stat['miss']:>9,}개"),
    ("검색 입력", f"문서당 최대 {MAX_CHARS:,}자"),
]
for k, v in rows:
    print(pad(k, 20) + v)
print("분야별 문서 수")
kv = [f"{k}\u00a0{v:,}" for k, v in kinds.most_common()]
print(wrap(" · ".join(kv), indent="  ").replace("\u00a0", " "))

## 2-1. 원문 잔존 검사

가명 코퍼스에 `original_text` 가 남았는지 문서마다 확인합니다.
세 가지는 유출이 아니므로 걸러 내고 셉니다.

- 대체어가 원문을 품은 라벨(`피해자` → `피해자-1`,
  `강릉시` → `강원도 강릉시 A`) — 일반명사이거나 상위 지역으로
  일반화한 결과입니다. 검사 대상에서 뺍니다.
- 다른 라벨의 대체어 속 글자(`군산시` 가 `전북특별자치도 군산시 B`
  안에 있는 경우) — 본문에서 **대체어를 걷어 낸 나머지**를 봅니다.
- `#이름#` — 판결문이 공개될 때 이미 가려진 자리입니다. 라벨이
  붙지 않은 판사·대리인 이름 자리에 그대로 남는데, 실명이 아니라
  가림 기호이므로 검사 대상에서 뺍니다.

원문 값 자체는 출력하지 않습니다. 건수만 셉니다.

In [ ]:
leaks, checked = 0, 0
bad_docs = []
by_id = {f.stem: f for f in label_files}
with CORPUS.open() as fp:
    for line in fp:
        rec = json.loads(line)
        doc = json.loads(by_id[rec["id"]].read_text())
        reps = {a["replacement_text"]
                for a in doc["annotations"]}
        rest = "\n".join([rec["head"], rec["full"]])
        for r in sorted(reps, key=len, reverse=True):
            rest = rest.replace(r, "\0")
        target = [a["original_text"]
                  for a in doc["annotations"]
                  if a["original_text"] != MASK
                  and a["original_text"]
                  not in a["replacement_text"]]
        n = sum(1 for o in target if o in rest)
        checked += len(target)
        if n:
            leaks += n
            bad_docs.append(rec["id"])

print(f"검사한 라벨   {checked:,}개")
print(f"원문 잔존     {leaks:,}건 · 문서 {len(bad_docs):,}건")
assert leaks == 0, "원문 문자열이 남았습니다"
print("검사 통과 — 가명 코퍼스에 원문 문자열 없음")

## 3. 임베딩 — bge-m3

`BAAI/bge-m3` (MIT) 로 문서마다 1,024차원 벡터를 만듭니다.
결과는 `WORK_DIR/emb_bge-m3.npy` 에 저장하고, 파일이 있으면
다시 계산하지 않습니다.

오래 걸립니다(M3 Pro 에서 16,800건에 약 1시간). 먼저 맛만 보려면
`LIMIT = 500` 처럼 줄이세요. 문서 수가 바뀌면 캐시를 새로 만듭니다.

다 끝나면 모델을 내리고 GPU 메모리를 비웁니다. 임베딩 모델과
5단계 생성 모델이 한 프로세스에서 GPU 를 동시에 잡으면 멈출 수
있어서, 검색 단계의 쟁점 임베딩은 CPU 로 합니다.

In [ ]:
import gc
import time

import numpy as np
from sentence_transformers import SentenceTransformer

EMB_MODEL = "BAAI/bge-m3"
LIMIT = None
EMB = WORK_DIR / "emb_bge-m3.npy"
IDS = WORK_DIR / "emb_ids.json"

with CORPUS.open() as fp:
    corpus = [json.loads(l) for l in fp]
if LIMIT:
    corpus = corpus[:LIMIT]
ids = [r["id"] for r in corpus]

cached = (EMB.exists() and IDS.exists()
          and json.loads(IDS.read_text()) == ids)
if cached:
    emb = np.load(EMB)
    print(f"캐시 사용 {emb.shape}")
else:
    model = SentenceTransformer(EMB_MODEL, device=DEV)
    if DEV != "cpu":
        model.half()
    t0 = time.perf_counter()
    emb = model.encode([r["text"] for r in corpus],
                       batch_size=16,
                       normalize_embeddings=True,
                       show_progress_bar=True)
    sec = time.perf_counter() - t0
    np.save(EMB, emb.astype(np.float32))
    IDS.write_text(json.dumps(ids))
    print(f"문서 {len(corpus):,}건 · 차원 {emb.shape[1]}")
    print(f"소요 {sec / 60:.1f}분 ({DEV})")
    del model

gc.collect()
if DEV == "mps":
    torch.mps.empty_cache()
elif DEV == "cuda":
    torch.cuda.empty_cache()

## 4. 쟁점 검색 — 상위 3건

`QUERY` 에 쟁점을 한 줄로 적습니다. 쟁점을 bge-m3 로 임베딩해
코퍼스 벡터와 코사인 유사도를 재고 가장 가까운 3건을 뽑습니다.
벡터를 정규화해 뒀으므로 내적이 곧 코사인 유사도입니다.

1·2·최종심 묶음은 한 사건의 심급별 판결이 따로 들어 있어,
같은 사건의 1심·2심이 나란히 뽑힐 수 있습니다.

In [ ]:
QUERY = ("제방부지 및 제외지가 법률 제2292호 하천법 "
         "개정법률 시행일(1971. 7. 20.)부터 법률 제3782호 "
         "하천법 중 개정법률의 시행일(1984. 12. 31.) 전에 "
         "국유로 된 경우, 명시적인 보상규정이 없더라도 "
         "관할관청이 소유자가 입은 손실을 보상하여야 "
         "하는지 여부")
TOP_K = 3

qmodel = SentenceTransformer(EMB_MODEL, device="cpu")
q = qmodel.encode([QUERY], normalize_embeddings=True)[0]
del qmodel
gc.collect()

scores = emb @ q
top = np.argsort(-scores)[:TOP_K]
hits = [corpus[i] for i in top]

print(wrap(f"쟁점: {QUERY}"))
for rank, i in enumerate(top, 1):
    r = corpus[i]
    gist = (r["holding"] or r["summary"] or r["reason"])
    gist = " ".join(gist.split())
    if len(gist) > 80:
        gist = gist[:80] + "…"
    print("-" * 64)
    print(f"[{rank}] 유사도 {scores[i]:.3f} · {r['kind']}")
    print(wrap(r["head"], indent="    "))
    print(wrap(gist, indent="    "))

## 5. 이유 초안 생성 — Qwen3-4B

아래 두 셀 중 기기에 맞는 하나만 실제로 돕니다.

- **Apple Silicon** — `mlx-community/Qwen3-4B-Instruct-2507-4bit`
  (Apache-2.0, 4비트 양자화, 약 2.3GB)
- **그 밖의 기기** — 같은 모델의 transformers 판
  `Qwen/Qwen3-4B-Instruct-2507` (Apache-2.0, 약 8GB).
  GPU 가 없으면 매우 느립니다.

모델은 처음 실행할 때 Hugging Face 에서 한 번 내려받습니다.
판결문은 이 컴퓨터 밖으로 나가지 않습니다.

In [ ]:
MAX_NEW = 700

if APPLE:
    from mlx_lm import generate as mlx_generate
    from mlx_lm import load
    from mlx_lm.sample_utils import make_sampler

    LLM = "mlx-community/Qwen3-4B-Instruct-2507-4bit"
    llm, tok = load(LLM)

    def chat(messages):
        prompt = tok.apply_chat_template(
            messages, add_generation_prompt=True)
        return mlx_generate(
            llm, tok, prompt=prompt, max_tokens=MAX_NEW,
            sampler=make_sampler(temp=0.0))

    print("loaded", LLM)

In [ ]:
if not APPLE:
    from transformers import AutoModelForCausalLM
    from transformers import AutoTokenizer

    LLM = "Qwen/Qwen3-4B-Instruct-2507"
    tok = AutoTokenizer.from_pretrained(LLM)
    llm = AutoModelForCausalLM.from_pretrained(
        LLM, torch_dtype="auto", device_map="auto")

    def chat(messages):
        ids = tok.apply_chat_template(
            messages, add_generation_prompt=True,
            return_tensors="pt").to(llm.device)
        out = llm.generate(ids, max_new_tokens=MAX_NEW,
                           do_sample=False)
        return tok.decode(out[0, ids.shape[1]:],
                          skip_special_tokens=True)

    print("loaded", LLM)

### 근거 있음 — 참고 판결 3건을 넣고 생성

검색한 3건의 사건 정보·판시사항·판결요지·이유 앞부분을 프롬프트에
넣고, 인용할 때 `[1]` 처럼 번호를 달게 합니다. 온도 0 으로 뽑아
같은 입력이면 같은 초안이 나옵니다.

In [ ]:
SYSTEM = ("당신은 한국 법원의 판결 이유 초안을 쓰는 보조자입니다. "
          "가상의 사건에 대한 초안이며 법률 자문이 아닙니다.")
TASK = ("다음 쟁점에 대한 판결 이유 초안을 한국어로 쓰세요. "
        "'1. 인정 사실', '2. 판단', '3. 결론' 세 부분으로 "
        "나누고 600자 안팎으로 쓰세요.")
REF_CHARS = 700


def ref_block(hits):
    parts = []
    for n, r in enumerate(hits, 1):
        body = "\n".join(x for x in [
            r["holding"], r["summary"],
            r["reason"][:REF_CHARS]] if x)
        parts.append(f"[{n}] {r['head']}\n{body}")
    return "\n\n".join(parts)


with_ref = chat([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": (
        f"{TASK} 아래 참고 판결만 근거로 삼고, 근거로 쓴 "
        "문장 끝에 [1]처럼 번호를 다세요. 참고 판결에 없는 "
        "법리는 지어내지 마세요.\n\n"
        f"쟁점: {QUERY}\n\n참고 판결\n{ref_block(hits)}")},
])
print(wrap(with_ref.strip()))

### 근거 없음 — 같은 지시, 참고 판결 없이 생성

모델이 제 지식만으로 쓰면 무엇이 달라지는지 비교합니다.
조문 번호·판례 번호를 그럴듯하게 지어내는지 눈여겨보세요.

In [ ]:
no_ref = chat([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": f"{TASK}\n\n쟁점: {QUERY}"},
])
print(wrap(no_ref.strip()))

## 정리

- 라벨을 적용한 가명 코퍼스만 임베딩·검색·생성에 썼습니다.
  원문 잔존 검사가 통과해야 다음 단계로 넘어갑니다.
- `WORK_DIR` 의 코퍼스·임베딩 파일은 원본과 같은 약관을 따릅니다.
  공유하거나 저장소에 올리지 마세요.
- 생성된 이유는 가상의 초안입니다. 인용 번호가 달려 있어도
  참고 판결 원문과 대조해 보세요.